# scBeacon: Prepared H5AD To Model To Figures

## Goal
Start from the prepared raw-count H5AD inputs retained on the server, train the full shared scBeacon model, and recompute pairs, QC, DEG, markers, enrichment and figures.

**This is not a frozen-table redraw.** No training result is prefilled. This new notebook has not yet been executed end-to-end; compute nodes were unavailable when it was authored. The existing manuscript-result notebook is a separate archived viewer, not evidence that these training cells ran.


## Setup
Activate the analysis environment and start Jupyter **inside a GPU compute allocation**. Do not execute Python on a login node. Install both requirements files and CUDA-capable PyTorch.

On the original server, set `SCBEACON_INPUT_ROOT` to `/home/kailu/links/projects/rrg-junding/kailu/scBeacon_codex/02_github_public_release/03_reproduction` before launching Jupyter. On another machine, point it to an unpacked copy containing `01_ipf/02_data`, `02_covid/02_data`, and `03_lung/02_data`.

Set `SCBEACON_RUN_ROOT` to a new output location with sufficient quota. Large H5AD inputs are not in the Git repository. Do not reselect HVGs or normalize the already prepared 2,000-feature input again.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scbeacon.py").is_file())
sys.path.insert(0, str(ROOT / "01_tool/03_runtime"))
from workflow import compute_only
from pipeline import PreparedInputPipeline
compute_only()

DATASET = "ipf"
INPUT_ROOT = Path(os.environ.get("SCBEACON_INPUT_ROOT", str(ROOT / "03_reproduction"))).expanduser()
RUN_PARENT = Path(os.environ.get("SCBEACON_RUN_ROOT", str(Path.home() / "scbeacon_new_runs"))).expanduser()
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUN_PARENT / DATASET / RUN_ID
USE_CUDA = True
# Explicitly allows submission of passing DEG symbols to live ToppGene later.
ALLOW_NETWORK = True
# None plots every new retained pair; never force a manuscript pair through QC.
DISPLAY_PAIR_LABELS = None
print("Prepared H5AD input root:", INPUT_ROOT)
print("NEW output root:", RUN_ROOT)


## Steps
### 1. Create a new run and validate actual H5AD inputs
The run configuration has no archived embedding or numerical-reference directory. The optional IPF gene-ID-to-symbol map is feature annotation only, not a DEG or enrichment result. Validation reads the two raw-count matrices and their combined normalized-log matrix and checks ordered cell/feature consistency.

If continuing an interrupted run, explicitly use `PreparedInputPipeline.open(previous_run_root)` instead, then start at its next unfinished stage. Do not call completed stages again or remove checkpoints.


In [ ]:
run = PreparedInputPipeline.create(
    DATASET, input_root=INPUT_ROOT, run_root=RUN_ROOT,
    use_cuda=USE_CUDA, display_pair_labels=DISPLAY_PAIR_LABELS,
)
input_checks = run.check_inputs()
display(pd.Series(input_checks, name="checked_from_H5AD"))
assert "embedding" not in run.config
assert "reference_dir" not in run.config
assert input_checks["embedding_shape"] is None


### 2. Train the full shared model
This cell **executes training**, not a preview: pretrain40 epochs, then all33 iterations with reconstruction20 and contrastive5 epochs per iteration. It calls the same shared model code as the user-facing tool. No cells are silently subsampled, no historical embedding is copied, and no short training schedule replaces the manuscript recipe.

This may take a long time. Watch `08_logs/train.log` for progress. A missing GPU or failed subprocess stops the notebook; it does not fall back to an old model.


In [ ]:
training_output = run.train()
new_embedding = np.load(run.embedding, mmap_mode="r")
expected_cells = input_checks["condition1_cells"] + input_checks["condition2_cells"]
assert new_embedding.shape == (expected_cells, 128)
assert np.isfinite(new_embedding).all()
print("Newly trained encoder embedding:", run.embedding)
print("Shape:", new_embedding.shape)


### 3. Recompute KL pairs and disease-versus-control DEG
The next cell passes the embedding just trained above to separate-condition Leiden clustering at resolution1. It recomputes centroid KL candidates and Scanpy Wilcoxon DEG using the prepared normalized-log expression, with health/control as the reference. Positive log2FC means disease-up. Later marker/enrichment selection uses adjusted p<.05 and log2FC>1.


In [ ]:
pair_output = run.pairs_deg()
pair_manifest = json.loads((pair_output / "run.json").read_text())
assert Path(pair_manifest["embedding"]).resolve() == run.embedding.resolve()
formal = pd.read_csv(pair_output / "formal_pairs.tsv", sep="\t")
display(formal.head(10))
deg_summary = pd.read_csv(pair_output / "02_deg/deg_unit_summary.tsv", sep="\t")
display(deg_summary.head(10))


### 4. Recompute the final clean-pair filter
Count cross-condition Euclidean kNN30 edges in the new encoder embedding, apply one-sided Fisher tests and separate BH corrections across all formal pairs in each direction, then retain both cluster sizes>=100 and both q<.05. Celltype whitelists and ATII-specific exceptions are not used. The new result is not forced to contain the archived number of pairs.


In [ ]:
qc_output = run.qc()
clean = pd.read_csv(run.clean_pairs, sep="\t")
print("New formal pairs:", len(formal), "New clean pairs:", len(clean))
display(clean[["pair_id", "pair_cluster_label", "reference_cluster_size", "group_cluster_size",
               "forward_fisher_qvalue_bh", "reverse_fisher_qvalue_bh"]])
if clean.empty:
    raise RuntimeError("This fit retained no clean pairs. Inspect the new tests; do not substitute a historical pair list.")


### 5. Compute marker expression and heatmap values
Use the newly computed DEG and clean pairs. Select up to5 eligible disease-up markers per pair, compute means from the actual normalized-log matrix, then gene-wise z-scores across displayed rows. Two rows represent each pair: control then disease. COVID host-marker display excludes VIRAL_ features; this is not host-only retraining or a change to the original DEG test universe.


In [ ]:
marker_output = run.markers()
marker_columns = pd.read_csv(marker_output / "heatmap_columns.tsv", sep="\t")
mean_expression = pd.read_csv(marker_output / "mean_expression.tsv", sep="\t")
display(marker_columns.head(15))
print("New mean-expression table shape:", mean_expression.shape)


### 6. Run fresh enrichment from all passing disease-up DEG
This cell calls the real ToppGene service for the newly retained pairs. It uses all mapped passing DEG, not only heatmap markers. Complete queries, mappings, missing genes and responses are saved under `06_enrichment`.

Requires explicit `ALLOW_NETWORK=True` and outbound HTTPS from this compute node or a personal workstation. If no network is available, stop here and explicitly reopen this same run in a suitable environment. Do not execute Python on the HPC login node, and do not load cached manuscript enrichment as if newly computed. Database updates can change terms and adjusted p-values.


In [ ]:
enrichment_output = run.enrich(allow_network=ALLOW_NETWORK)
summary_paths = sorted(enrichment_output.glob("*/08_toppgene_summary.tsv"))
enrichment_summary = pd.concat([pd.read_csv(p, sep="\t") for p in summary_paths], ignore_index=True)
display(enrichment_summary[["unit_id", "input_gene_count", "mapped_gene_count", "n_annotations", "status"]])


### 7. Generate figures from this run
Recompute UMAP using cosine neighbors10, spectral initialization and min_dist0.8. The marker heatmap and Reactome+WikiPathways plots use the new tables generated above. Archived coordinates, p-values and fixed display-pair selections are disabled. New disease-relevance highlights are not invented automatically; input celltype annotations are used for display.

Each scientific plot is exported separately as PDF/SVG/PNG; vector text/lines and 300dpi rasterized UMAP clouds. Fresh-run layouts can differ from the manuscript's Illustrator assembly.


In [ ]:
figure_output = run.plot()
source_values = figure_output / "01_source_values"
panels = figure_output / "02_individual_panels"
figure_manifest = json.loads((figure_output / "run_manifest.json").read_text())
assert figure_manifest["mode"] == "new_scientific_run"
assert figure_manifest["umap_recomputed"] is True
print("NEW source tables:", source_values)
print("NEW separate panel files:", panels)


### 8. Inspect newly generated UMAPs
These image files were generated by the preceding stages in RUN_ROOT, not copied from the archived manuscript panels.


In [ ]:
for image in sorted(panels.glob("*_umap_*.png")):
    print(image.name)
    display(Image(filename=str(image)))


### 9. Inspect the new clean-pair network and marker heatmap


In [ ]:
display(Image(filename=str(panels / f"{DATASET}_clean_pair_network.png")))
display(Image(filename=str(panels / f"{DATASET}_marker_heatmap.png")))


### 10. Inspect current pathway terms
All current selected pathways are saved. The notebook previews at most two pair figures to keep the page readable; enrichment and panel export are not restricted to these two pairs. An absence of significant terms remains an absence, not a trigger to substitute older results.


In [ ]:
new_terms = pd.read_csv(source_values / "selected_pathways.tsv", sep="\t")
display(new_terms.head(20))
for image in sorted(panels.glob("*_pathways_*.png"))[:2]:
    print(image.name)
    display(Image(filename=str(image)))


## Checks
Confirm each real subprocess completed, that the formal-pair stage used this run's trained embedding, and that new heatmap rows belong to new clean pairs. Full command lines and failure logs are retained. Do not interpret pair arrows as proof of cell lineage conversion or cell-level p-values as donor-level validation.


In [ ]:
statuses = [json.loads((RUN_ROOT / "08_logs" / f"{stage}.json").read_text()) for stage in run.STAGES]
assert all(s["status"] == "completed" for s in statuses)
assert all(s["archived_numerical_fallback"] is False for s in statuses)
rows = pd.read_csv(source_values / "heatmap_rows.tsv", sep="\t")
assert len(rows) == 2 * len(clean)
assert set(rows.pair_id) <= set(clean.pair_id)
display(pd.DataFrame(statuses)[["stage", "status", "duration_seconds", "output"]])
print("Completed this run from prepared H5AD inputs through new model training and new figures.")


## Next Steps
Save the executed notebook under your new run folder to retain genuine training-derived outputs. Review full DEG, Fisher tests and enrichment responses before choosing manuscript panels. Keep the original prepared inputs and historical results unchanged. Use the separate archived-result notebooks only when you explicitly want the historical numerical figure values.
